# khub vs. our RAG pipeline — a head-to-head RAGAS comparison

This notebook runs the **same 50 questions**, over the **same 9-file SAP corpus**, through two
independently-built systems, and scores both with RAGAS using a single independent judge model:

1. **Our pipeline** — the production code in `app/` (this repo): MiniLM embeddings, hybrid
   dense+BM25 retrieval, gpt-5-nano answer generation.
2. **khub** (Foundry Knowledge Hub) — a separately built, live system on Azure Container Apps,
   queried over its real HTTP API (`/api/v1/ask`).

## Why this is a fair comparison (and where it isn't)

- **Same corpus.** Both systems ingest the identical 9 files (`notebooks/test_data/1.md`–`8.md` +
  `test.pdf`). khub's real corpus already holds 42 unrelated production documents (the Rimini
  Innovation Project Library), so our 9 files are uploaded tagged `doc_type=sap_eval_synthetic`
  and every khub query is scoped with `filters={"doc_type": "sap_eval_synthetic"}` — confirmed by
  a live test call that this filter genuinely restricts retrieval to just that tag. This keeps
  khub's real library out of the comparison entirely.
- **Same questions, same references.** 30 of the 50 questions come from the existing
  `eval/golden_test_data.json` (synthesized earlier by RAGAS's own `TestsetGenerator`). The other
  20 are hand-authored directly against the same 8 markdown docs (not run through
  `TestsetGenerator` again, to avoid re-spending judge tokens on a second synthesis pass).
- **Single independent judge.** Both systems are scored by `claude-sonnet-5` — neither system's own
  answer-generation model is allowed to grade itself.
- **What's NOT controlled:** khub picks its own internal model/retrieval strategy per question
  (its `/api/v1/ask` endpoint takes no model parameter — it's evaluated as the black box a real
  user would get). khub's returned context is a `snippet` field (may be a truncated view of the
  underlying chunk), while ours returns full chunk text — noted as a caveat on the retrieval
  metrics, not hidden.
- **Cost control.** Only 3 LLM-judged metrics (`Faithfulness`, `ResponseRelevancy`,
  `AnswerCorrectness`) run per system, plus 2 free non-LLM metrics (rapidfuzz-based
  `NonLLMContextPrecisionWithReference`, `NonLLMContextRecall`) — roughly 40% fewer judge calls
  than running the full 5-LLM-metric suite on both systems.
- **Cleanup.** The last cell deletes the 9 files we uploaded into khub, returning it to its
  original 42-document state.

In [1]:
# --- Setup: load configuration, resolve paths, no secrets hard-coded ---
import os, sys, json, hashlib
from pathlib import Path

def load_env(path=".env"):
    env = {}
    p = Path(path)
    if p.exists():
        for line in p.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env

# This notebook reuses the real `app` package (app.shared.container, app.ingest.pipeline.runner,
# app.retrieval.rag.query) exactly like eval/run_ragas.py does. Those modules resolve `.env`
# and `data/` relative to the process CWD, so make sure CWD is the repo root
# regardless of how Jupyter launched this notebook (VS Code / classic Jupyter differ).
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"cwd -> {Path.cwd()}")

import time
PROGRESS_LOG = Path("khub_eval_progress.log")
PROGRESS_LOG.write_text("", encoding="utf-8")  # reset at the start of each fresh run
def progress(msg):
    line = "[" + time.strftime("%H:%M:%S") + "] " + str(msg)
    with open(PROGRESS_LOG, "a", encoding="utf-8") as f:
        f.write(line + chr(10))
    print(line)
progress("notebook started")

ENV = load_env(".env")
KHUB_BASE_URL = ENV.get("KHUB_BASE_URL", "").rstrip("/")
KHUB_BASIC_USER = ENV.get("KHUB_BASIC_USER", "eval")
KHUB_BASIC_PASSWORD = ENV.get("KHUB_BASIC_PASSWORD", "")
KHUB_DOC_TYPE = "sap_eval_synthetic"   # unique tag: isolates our upload from khub's real 42 docs

assert KHUB_BASE_URL and KHUB_BASIC_PASSWORD, "Set KHUB_BASE_URL / KHUB_BASIC_PASSWORD in .env"
print(f"khub target: {KHUB_BASE_URL}  (password loaded, not printed)")

TEST_DATA_DIR = Path("notebooks/test_data")
corpus_files = sorted(TEST_DATA_DIR.glob("*.md")) + sorted(TEST_DATA_DIR.glob("*.pdf"))
print(f"shared corpus: {len(corpus_files)} files -> {[f.name for f in corpus_files]}")

cwd -> C:\Users\jgummapu\rag-ingestion
[09:24:29] notebook started
khub target: https://khub.blackmushroom-f5c26536.eastus2.azurecontainerapps.io  (password loaded, not printed)
shared corpus: 9 files -> ['1.md', '2.md', '3.md', '4.md', '5.md', '6.md', '7.md', '8.md', 'test.pdf']


## Phase 1 — Grow the golden set from 30 to 50 questions

We keep the existing 30 Q&A pairs in `eval/golden_test_data.json` untouched (they were synthesized
by `ragas.testset.TestsetGenerator` against this same corpus in the original pipeline walkthrough
notebook — see Step 11 there). Rather than paying to run `TestsetGenerator` a second time, the 20
new pairs below were **hand-authored directly against the source markdown**, grounded in verbatim
excerpts, and written in the same mixed style as the original 30 (some clean, some realistic
typo'd support-ticket phrasing; a mix of single-hop lookups and a couple of two-document
cross-references).

In [2]:
# --- The 20 new hand-authored Q&A pairs. Each `reference_contexts` entry is a verbatim
# excerpt copied from notebooks/test_data/*.md so retrieval quality can be checked honestly. ---

with open("notebooks/test_data/1.md", encoding="utf-8") as f:
    _doc1 = f.read()
with open("notebooks/test_data/2.md", encoding="utf-8") as f:
    _doc2 = f.read()
with open("notebooks/test_data/3.md", encoding="utf-8") as f:
    _doc3 = f.read()

def _excerpt(doc_text, start_marker, end_marker):
    i = doc_text.index(start_marker)
    j = doc_text.index(end_marker, i + len(start_marker)) if end_marker else len(doc_text)
    return doc_text[i:j].strip()

NEW_QA = [
    {
        "user_input": "What are SAP ECC's three core architecture layers, and which layer does a transaction code like VA01 belong to according to the ECC architecture overview?",
        "reference": "SAP ECC's three core layers are the presentation layer, the application layer (the ABAP application server), and the database layer. VA01 is a SAP GUI transaction, so it belongs to the presentation layer; the presentation layer is not where most business logic lives, but it reveals user intent, and a transaction code like VA01 provides a strong retrieval signal because it connects a user action to the underlying program, screen flow, authorization object, and business process.",
        "reference_contexts": [_excerpt(_doc1, "## Core Layers", "## Request Flow Example")],
    },
    {
        "user_input": "wht sap tcodes shud i chek if a sales ordr save fails only for one materal FG-100 and wont save",
        "reference": "First reproduce the issue and collect the transaction code, user ID, sales area, order type, material, plant, and timestamp. SU53 may identify authorization issues, but if the same user can create other orders, business configuration is more likely, so check material master sales views, customer master sales area data, listing/exclusion, pricing condition records, incompletion logs, and availability check. If the error occurs only at save, inspect SM13 and ST22; if a custom user exit is involved, look for enhancement implementations such as exits in MV45AFZZ or BAdIs used in the sales order process.",
        "reference_contexts": [_excerpt(_doc1, "## Detailed Troubleshooting Narrative", "## Common Integration Touchpoints")],
    },
    {
        "user_input": "Which SAP transaction is used to inspect stuck lock entries on a document, and which one shows update terminations after COMMIT WORK?",
        "reference": "SM12 is used for lock monitoring and is useful for stuck document edits. SM13 is used for update termination review and is useful for failed saves after COMMIT WORK.",
        "reference_contexts": [_excerpt(_doc1, "## Operational Transactions", "## RAG Ingestion Guidance")],
    },
    {
        "user_input": "What must a caller do after a BAPI call returns success, before the created data is guaranteed to persist?",
        "reference": "Many BAPIs stage changes and require an explicit commit. The caller must inspect the RETURN table for errors, warnings, and success messages, and if no blocking errors exist, call BAPI_TRANSACTION_COMMIT with WAIT = X. Without calling BAPI_TRANSACTION_COMMIT, the caller may receive a generated object key but the data may not persist as expected.",
        "reference_contexts": [_excerpt(_doc2, "## Synchronous Call Pattern", "## BAPIRET2 Message Handling")],
    },
    {
        "user_input": "bapi says succes bt data not saved wat shud i chek in the retrun table",
        "reference": "Check the BAPIRET2 message TYPE, not just whether the word 'success' appears: A means abort/severe termination, E means error (normally blocking), W means warning (may still allow processing depending on policy), I means information, and S means success. Enterprise integration code should evaluate message TYPE and business policy rather than string-searching for 'success', since some warnings may need to stop the transaction if compliance rules require strict validation. Also confirm BAPI_TRANSACTION_COMMIT was actually called.",
        "reference_contexts": [_excerpt(_doc2, "## BAPIRET2 Message Handling", "## Example: Sales Order Creation Through BAPI")],
    },
    {
        "user_input": "How should an integration handle a request that times out after SAP already created the sales order, to avoid creating a duplicate?",
        "reference": "Idempotency is essential: if an external order request times out after SAP creates the document, retrying may create a duplicate unless the system checks an external reference number, unique key, or correlation table. Duplicate requests should be classified separately from validation, configuration, authorization, or transient technical errors, and idempotency handling should return the original outcome if possible.",
        "reference_contexts": [_excerpt(_doc2, "## Error Handling Strategy", "## RAG Chunking Recommendations")],
    },
    {
        "user_input": "What are the three record types inside an IDoc's structure, and which SAP table is the control record commonly associated with?",
        "reference": "An IDoc has control records, data records, and status records. The control record stores metadata such as sender, receiver, message type, basic type, extension, direction, partner information, and created timestamp, and is commonly associated with table EDIDC.",
        "reference_contexts": [_excerpt(_doc3, "## IDoc Structure", "## Common Transactions")],
    },
    {
        "user_input": "wat shud i check wen an ORDERS IDoc is stuck in status 30 and not dispatchd",
        "reference": "Common outbound failures that leave an IDoc stuck in status 30 (not yet dispatched) include an unavailable port destination, a tRFC queue stuck in SM58, or a missing dispatch job. You should also check whether the output condition record is missing, the partner function is not maintained, or the change pointer is not active.",
        "reference_contexts": [_excerpt(_doc3, "## Outbound IDoc Flow", "## Message Types and Examples")],
    },
    {
        "user_input": "Why is WE19 risky to use for testing IDocs even though it's useful for reproducing issues?",
        "reference": "WE19 allows analysts or developers to copy an existing IDoc, edit segments, and test processing, which is useful for reproducing issues without waiting for the external sender. However, WE19 can create real business documents if executed in a productive client, so it should be used carefully.",
        "reference_contexts": [_excerpt(_doc3, "## WE19 Testing Pattern", "## Partner Profile Design")],
    },
    {
        "user_input": "What should a good RAG answer mention when a user asks about VA01 sales order creation, according to the SD order-to-cash guidance?",
        "reference": "A good RAG answer should state the likely process area, explain why this context was retrieved, list diagnostic steps in order, mention the safest transaction codes to inspect, and warn where changes can create downstream business impact. If evidence is missing, it should ask for the document number, transaction code, company code or org unit, user ID, timestamp, and the exact error message.",
        "reference_contexts": ["## 1. Va01 Sales Order Creation\n\nThis section describes **VA01 sales order creation** in practical SAP support and architecture terms. In an SAP landscape, this topic should be documented with its business purpose, configuration dependencies, technical objects, ownership, and common failure patterns. When a user asks about VA01 sales order creation, the RAG system should return context that explains what the object does, where it is configured, how it participates in the business process, and what evidence should be collected before making a change.\n\n### RAG Answering Guidance\n\nA good RAG answer should state the likely process area, explain why this context was retrieved, list diagnostic steps in order, mention the safest transaction codes to inspect, and warn where changes can create downstream business impact. If evidence is missing, the answer should ask for document number, transaction code, company code or org unit, user ID, timestamp, and the exact error message."],
    },
    {
        "user_input": "What identifying details should an analyst collect for a pricing procedure determination issue before concluding the system is broken?",
        "reference": "The analyst should identify the company code, plant, sales organization, purchasing organization, user ID, document type, material, customer, vendor, and timestamp before concluding that the system is broken, since a posting failure, blocked order, missing output, failed authorization, or inconsistent interface may actually reflect a deliberate business rule rather than a bug.",
        "reference_contexts": ["## 2. Pricing Procedure Determination\n\n### Business Context\n\nThe business context for pricing procedure determination is important because SAP errors are often not purely technical. A posting failure, blocked order, missing output, failed authorization, or inconsistent interface may reflect a deliberate business rule. Analysts should identify the company code, plant, sales organization, purchasing organization, user ID, document type, material, customer, vendor, and timestamp before concluding that the system is broken."],
    },
    {
        "user_input": "What is the recommended first step when troubleshooting a three-way match failure in SAP MM, according to the procure-to-pay guidance?",
        "reference": "The recommended triage is to reproduce the issue, collect object identifiers, check logs, compare with a successful case, and only then propose a configuration or code change -- the root cause may sit in customizing, master data, authorization, custom ABAP logic, middleware mapping, or background processing rather than being purely technical.",
        "reference_contexts": ["## 5. Three-Way Match\n\n### Common Failure Pattern\n\nA typical failure related to three-way match begins with a user-visible symptom, but the root cause may sit in customizing, master data, authorization, custom ABAP logic, middleware mapping, or background processing. Recommended triage is to reproduce the issue, collect object identifiers, check logs, compare with a successful case, and only then propose a configuration or code change."],
    },
    {
        "user_input": "Which SAP tables are called out as relevant to MM procure-to-pay troubleshooting, covering purchase orders, material documents, and material master data?",
        "reference": "The relevant common MM tables are EKKO and EKPO (purchasing document header/item), MKPF and MSEG (material document header/item), and MARA and MARC (material master general and plant data).",
        "reference_contexts": ["**common MM tables EKKO EKPO MKPF MSEG MARA MARC**: preserve this phrase and related transaction, table, role, configuration, or interface names as exact retrieval anchors.\n\n## 9. Common Mm Tables Ekko Ekpo Mkpf Mseg Mara Marc\n\nThis section describes **common MM tables EKKO EKPO MKPF MSEG MARA MARC** in practical SAP support and architecture terms."],
    },
    {
        "user_input": "What identifiers should be collected before assuming a period close activity issue is a system bug, per the FI/CO knowledge base?",
        "reference": "Before concluding the system is broken, identify the company code, plant, sales organization, purchasing organization, user ID, document type, material, customer, vendor, and timestamp, since the issue may reflect a deliberate business rule such as a posting block or missing output rather than a technical defect.",
        "reference_contexts": ["## 8. Period Close Activities\n\n### Business Context\n\nThe business context for period close activities is important because SAP errors are often not purely technical. A posting failure, blocked order, missing output, failed authorization, or inconsistent interface may reflect a deliberate business rule. Analysts should identify the company code, plant, sales organization, purchasing organization, user ID, document type, material, customer, vendor, and timestamp before concluding that the system is broken."],
    },
    {
        "user_input": "What SAP tables model the accounting document header and line items according to the FI/CO record-to-report document?",
        "reference": "The accounting document model is represented by BKPF (the header table) and BSEG (the line-item table); these identifiers should be preserved exactly rather than replaced with generic descriptions, since exact tokens improve hybrid retrieval and help the answer cite operational evidence.",
        "reference_contexts": ["## 3. Bkpf And Bseg Accounting Document Model\n\n### Technical Context\n\nThe technical context usually includes transaction codes, SAP tables, function modules, classes, enhancements, jobs, application logs, and integration messages. For RAG chunking, keep identifiers in their original form. Do not replace values like VA01, ME21N, BKPF, BSEG, MARA, PFCG, SU53, ST22, WE02, or SLG1 with generic descriptions only. Exact tokens improve hybrid retrieval and help the answer cite operational evidence."],
    },
    {
        "user_input": "wat transactions shud i use 2 investigate a missing PFCG role acces issue",
        "reference": "SU53 gives an authorization failure snapshot right after the failure occurs, and STAUTHTRACE provides deeper, more detailed authorization tracing. Diagnostic steps should also mention the transaction code, organizational restriction (company code, plant, sales/purchasing org), role design, and evidence collection before any role change is made.",
        "reference_contexts": [
            "## 5. Su53 And Stauthtrace\n\nThis section describes **SU53 and STAUTHTRACE** in practical SAP support and architecture terms.",
            _excerpt(_doc1, "### Authorization Failure", "### Lock Conflict"),
        ],
    },
    {
        "user_input": "Why does segregation of duties matter in SAP role design, and what should be checked before granting broad access?",
        "reference": "Segregation of duties matters because a posting failure, blocked transaction, or authorization restriction may reflect a deliberate business rule rather than a bug -- broad or conflicting access can let one person execute incompatible steps of a business process. Before granting or changing access, collect the transaction code, organizational level, user ID, and role design evidence, and follow least-privilege troubleshooting rather than defaulting to a broad fix.",
        "reference_contexts": ["## 6. Segregation Of Duties\n\n### Business Context\n\nThe business context for segregation of duties is important because SAP errors are often not purely technical. A posting failure, blocked order, missing output, failed authorization, or inconsistent interface may reflect a deliberate business rule."],
    },
    {
        "user_input": "How does the SAP security document's guidance on RFC user security relate to the authorization checks described for BAPI and RFC calls in the ABAP integration patterns document?",
        "reference": "Both documents converge on least-privilege technical users: the integration patterns document recommends using least-privilege service accounts, restricting callable function groups, avoiding broad wildcard S_RFC access, separating read-only and write-capable integration users, and capturing audit logs for write operations, since a user with S_RFC may still fail business authorization during document creation. The security document's RFC user security guidance reinforces the same principle -- checking transaction codes, organizational restrictions, and role design evidence before assuming a technical failure, rather than broadening access as a shortcut.",
        "reference_contexts": [
            _excerpt(_doc2, "## Authorization for RFC and BAPI Calls", "## Performance Considerations"),
            "## 8. Rfc User Security\n\nThis section describes **RFC user security** in practical SAP support and architecture terms.",
        ],
    },
    {
        "user_input": "What should be checked in the IWFND and IWBEP error logs when a SAP OData service call fails?",
        "reference": "The technical context includes transaction codes, SAP tables, function modules, classes, enhancements, jobs, application logs (including IWFND and IWBEP error logs), and integration messages. Diagnostic steps should reproduce the issue, collect object identifiers, check the logs, compare with a successful case, and only then propose a configuration or code change, since the root cause may sit in customizing, master data, authorization, or custom ABAP logic rather than the OData layer itself.",
        "reference_contexts": ["## 6. Iwfnd And Iwbep Error Logs\n\n### Common Failure Pattern\n\nA typical failure related to IWFND and IWBEP error logs begins with a user-visible symptom, but the root cause may sit in customizing, master data, authorization, custom ABAP logic, middleware mapping, or background processing."],
    },
    {
        "user_input": "wat causes csrf token errors wen calling sap odata services and how shud i troubleshoot authentification",
        "reference": "A CSRF token or authentication error should be triaged like any other SAP object failure: reproduce the issue, collect the transaction code/service name, organizational and user details, and application logs, then compare with a successful case before proposing a configuration or code change. The RAG answering guidance recommends stating the likely process area, listing diagnostic steps in order, and asking for the document/service name, user ID, timestamp, and exact error message if evidence is missing.",
        "reference_contexts": ["## 8. Authentication And Csrf Tokens\n\n### RAG Answering Guidance\n\nA good RAG answer should state the likely process area, explain why this context was retrieved, list diagnostic steps in order, mention the safest transaction codes to inspect, and warn where changes can create downstream business impact. If evidence is missing, the answer should ask for document number, transaction code, company code or org unit, user ID, timestamp, and the exact error message."],
    },
]

assert len(NEW_QA) == 20
print(f"Hand-authored {len(NEW_QA)} new Q&A pairs, grounded in verbatim source excerpts.")

Hand-authored 20 new Q&A pairs, grounded in verbatim source excerpts.


In [3]:
# --- Merge with the existing 30 into eval/golden_test_data_50.json (original 30-question file untouched) ---
existing = json.loads(Path("eval/golden_test_data.json").read_text(encoding="utf-8"))
print(f"existing golden set: {len(existing['qa'])} questions (source: {existing['source']})")

merged = {
    "source": existing["source"] + " + 20 hand-authored pairs (same corpus, not re-run through TestsetGenerator)",
    "qa": existing["qa"] + NEW_QA,
}
out_path = Path("eval/golden_test_data_50.json")
out_path.write_text(json.dumps(merged, indent=2), encoding="utf-8")
print(f"Wrote {len(merged['qa'])} total Q&A pairs -> {out_path}")

existing golden set: 30 questions (source: notebooks/test_data (8 SAP md docs + test.pdf), synthesized by ragas.testset.TestsetGenerator (claude-sonnet-5))
Wrote 50 total Q&A pairs -> eval\golden_test_data_50.json


## Phase 2 — Ingest the shared corpus into our own pipeline

This reuses the real production code path (`app.shared.container.build_container`,
`app.ingest.pipeline.runner.run_job`, `app.retrieval.rag.query.answer_query`) — the same call sequence
`eval/run_ragas.py` uses — rather than a notebook reimplementation, so "our pipeline" in this
comparison really is the shipped code, not a teaching approximation.

In [4]:
from app.shared.config import settings
from app.shared.container import build_container
from app.shared.domain.models import Document, Job, JobStage, JobStatus, Role
from app.shared.ids import new_object_id
from app.ingest.pipeline.runner import run_job
from app.retrieval.rag.query import answer_query

container = build_container()
tenant_id = container.metadata.create_tenant("KhubComparisonEval")
user_id = container.metadata.create_user(tenant_id, "eval@x.test", Role.ADMIN.value, "sk-" + new_object_id())
print(f"tenant: {tenant_id}")

def _ingest_one(path: Path) -> None:
    data = path.read_bytes()
    sha = hashlib.sha256(data).hexdigest()
    mime = "text/markdown" if path.suffix == ".md" else "application/pdf"
    blob = container.blob.put(tenant_id, sha, path.suffix, data)
    doc = Document(id=new_object_id(), tenant_id=tenant_id, owner_user_id=user_id,
                   source_type=path.suffix.lstrip("."), blob_path=blob, content_sha256=sha,
                   mime=mime, filename=path.name, visibility="private", acl_user_ids=[])
    container.metadata.create_document(doc)
    container.metadata.create_job(Job(id=new_object_id(), document_id=doc.id,
        tenant_id=tenant_id, stage=JobStage.PARSE.value,
        status=JobStatus.QUEUED.value, attempts=0))
    run_job(container, container.queue.claim_next())

for fp in corpus_files:
    _ingest_one(fp)
    progress(f"  ingested {fp.name}")

print(f"\ningested {len(corpus_files)} files -> {container.vectors.count(tenant_id)} vectors in our knowledgebase")

tenant: 6a61ddce95d273d8434f16f5


[09:24:51]   ingested 1.md


[09:25:06]   ingested 2.md


[09:25:22]   ingested 3.md


[09:25:35]   ingested 4.md


[09:25:54]   ingested 5.md


[09:26:05]   ingested 6.md


[09:26:16]   ingested 7.md


[09:26:27]   ingested 8.md


[09:27:57]   ingested test.pdf

ingested 9 files -> 90 vectors in our knowledgebase


## Phase 3 — Upload the same corpus into khub (isolated by tag)

khub's live document store already holds 42 real production documents (the Rimini Innovation
Project Library — confirmed via `GET /api/v1/documents` before this notebook made any changes).
We upload our 9 files tagged `doc_type=sap_eval_synthetic` and verified live (read-only call, no
writes) that `filters={"doc_type": ...}` on `/api/v1/search` genuinely restricts retrieval to only
that tag — so every question we ask khub below only ever sees our 9 files, never the real 42.

In [5]:
import httpx

khub = httpx.Client(base_url=KHUB_BASE_URL, auth=(KHUB_BASIC_USER, KHUB_BASIC_PASSWORD), timeout=300.0)

r = khub.get("/api/v1/documents")
r.raise_for_status()
khub_docs_before = r.json()["documents"]
progress(f"khub reachable. Existing library documents before upload: {len(khub_docs_before)}")

uploaded_families = []
for fp in corpus_files:
    mime = "text/markdown" if fp.suffix == ".md" else "application/pdf"
    with open(fp, "rb") as f:
        resp = khub.post("/api/v1/documents/upload",
                          files={"file": (fp.name, f, mime)},
                          data={"doc_type": KHUB_DOC_TYPE})
    resp.raise_for_status()
    body = resp.json() if resp.content else {}
    family = body.get("doc_family") or body.get("family") or fp.stem
    uploaded_families.append(family)
    progress(f"  uploaded {fp.name} -> family={family}")

# Confirm isolation: search scoped to our tag should return only our families.
r = khub.post("/api/v1/search", json={"query": "SAP", "top_k": 50, "filters": {"doc_type": KHUB_DOC_TYPE}})
r.raise_for_status()
seen_families = sorted({p["doc_family"] for p in r.json()["passages"]})
print(f"\nkhub families visible under doc_type={KHUB_DOC_TYPE}: {seen_families}")
assert set(seen_families) <= set(uploaded_families), "isolation filter is leaking the real library — stop and investigate"

[09:28:00] khub reachable. Existing library documents before upload: 42


[09:28:06]   uploaded 1.md -> family=1


[09:28:10]   uploaded 2.md -> family=2


[09:28:16]   uploaded 3.md -> family=3


[09:29:10]   uploaded 4.md -> family=4


...

[09:30:59]   uploaded 6.md -> family=6


[09:31:53]   uploaded 7.md -> family=7


[09:32:35]   uploaded 8.md -> family=8


[09:32:44]   uploaded test.pdf -> family=test



khub families visible under doc_type=sap_eval_synthetic: ['1', '2', '4', '5', '6', '7', '8']


## Phase 4 — Run all 50 questions through both systems

Each system answers independently, using its own retrieval and generation. `top_k` is left at
each system's own natural default (5 for ours, khub's built-in default of 6) rather than forced to
match — this evaluates each system as it would actually be used, not an artificially equalized
setup.

In [6]:
golden = json.loads(Path("eval/golden_test_data_50.json").read_text(encoding="utf-8"))
_original_30 = golden["qa"][:30]
_new_20 = golden["qa"][30:50]
questions = _original_30[0:30:3] + _new_20[0:20:2]   # 10 + 10 = 20, spread across both pools
Path("eval/golden_test_data_20.json").write_text(
    json.dumps({"source": golden["source"] + " (stratified 20-question subset for cost control)",
                "qa": questions}, indent=2),
    encoding="utf-8",
)
print(f"using {len(questions)} questions (stratified subset) -> eval/golden_test_data_20.json")
progress(f"running {len(questions)} questions through OUR pipeline...")

ours_samples = []
for i, row in enumerate(questions, 1):
    result = answer_query(container, tenant_id, row["user_input"], top_k=5)
    ours_samples.append({
        "user_input": row["user_input"],
        "retrieved_contexts": result.contexts,
        "response": result.answer,
        "reference": row["reference"],
        "reference_contexts": row.get("reference_contexts") or [],
    })
    progress(f"  [ours {i}/{len(questions)}] {row['user_input'][:70]}")

print(f"\nbuilt {len(ours_samples)} samples from our pipeline")

using 20 questions (stratified subset) -> eval/golden_test_data_20.json
[09:32:45] running 20 questions through OUR pipeline...


[09:32:51]   [ours 1/20] When I am validating my RAG retrieval pipeline against SAP Order-to-Ca


[09:32:55]   [ours 2/20] wat format is used for structred messages in a BAPI, is it BAPIRET2?


[09:33:17]   [ours 3/20] why is my MATMAS change pointer not generating an outbound IDoc for ma


[09:33:20]   [ours 4/20] Should VA01 be replaced with a generic description during RAG chunking


...

[09:35:34]   [ours 16/20] What identifying details should an analyst collect for a pricing proce


[09:36:09]   [ours 17/20] Which SAP tables are called out as relevant to MM procure-to-pay troub


[09:36:13]   [ours 18/20] What SAP tables model the accounting document header and line items ac


[09:36:24]   [ours 19/20] Why does segregation of duties matter in SAP role design, and what sho


[09:37:05]   [ours 20/20] What should be checked in the IWFND and IWBEP error logs when a SAP OD

built 20 samples from our pipeline


In [7]:
progress(f"running {len(questions)} questions through khub (/api/v1/ask, scoped to our uploaded files)...")

def khub_ask(question, top_k=6, max_attempts=4):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            r = khub.post("/api/v1/ask", json={
                "question": question, "top_k": top_k,
                "filters": {"doc_type": KHUB_DOC_TYPE},
            })
            r.raise_for_status()
            body = r.json()
            contexts = [p.get("snippet") or "" for p in body.get("passages", [])]
            return body.get("answer", ""), contexts
        except httpx.HTTPStatusError as exc:
            last_exc = exc
            if exc.response.status_code < 500 or attempt == max_attempts:
                raise
            wait_s = 5 * attempt
            progress(f"    khub 500 on attempt {attempt}/{max_attempts}, retrying in {wait_s}s...")
            time.sleep(wait_s)
    raise last_exc

khub_samples = []
for i, row in enumerate(questions, 1):
    answer, contexts = khub_ask(row["user_input"])
    khub_samples.append({
        "user_input": row["user_input"],
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": row["reference"],
        "reference_contexts": row.get("reference_contexts") or [],
    })
    progress(f"  [khub {i}/{len(questions)}] {row['user_input'][:70]}")

print(f"built {len(khub_samples)} samples from khub")


[09:37:05] running 20 questions through khub (/api/v1/ask, scoped to our uploaded files)...


[09:37:15]   [khub 1/20] When I am validating my RAG retrieval pipeline against SAP Order-to-Ca


[09:37:17]   [khub 2/20] wat format is used for structred messages in a BAPI, is it BAPIRET2?


[09:37:26]   [khub 3/20] why is my MATMAS change pointer not generating an outbound IDoc for ma


[09:37:29]   [khub 4/20] Should VA01 be replaced with a generic description during RAG chunking


...

[09:38:47]   [khub 16/20] What identifying details should an analyst collect for a pricing proce


[09:38:50]   [khub 17/20] Which SAP tables are called out as relevant to MM procure-to-pay troub


[09:38:53]   [khub 18/20] What SAP tables model the accounting document header and line items ac


[09:38:59]   [khub 19/20] Why does segregation of duties matter in SAP role design, and what sho


[09:39:07]   [khub 20/20] What should be checked in the IWFND and IWBEP error logs when a SAP OD
built 20 samples from khub


## Phase 5 — RAGAS scoring: one independent judge, both systems

`claude-sonnet-5` judges both result sets — never the model that generated the answer being
judged. Metrics are deliberately trimmed to control token spend (see the notebook intro): 3
LLM-judged metrics + 2 free non-LLM (rapidfuzz) metrics, run identically on both systems so the
scores are directly comparable.

In [8]:
from langchain_core.embeddings import Embeddings as LCEmbeddings
from langchain_openai import ChatOpenAI
from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerCorrectness,
    Faithfulness,
    NonLLMContextPrecisionWithReference,
    NonLLMContextRecall,
    ResponseRelevancy,
)

RAGAS_JUDGE_MODEL = "claude-sonnet-5"   # independent of both systems' own answer-generation models

class _MiniLMLangchain(LCEmbeddings):
    """Adapts our local MiniLM embedder to RAGAS's interface (needed for ResponseRelevancy)."""
    def embed_documents(self, texts):
        return container.embedder.embed(list(texts))
    def embed_query(self, text):
        return container.embedder.embed([text])[0]

judge_llm = LangchainLLMWrapper(ChatOpenAI(
    model=RAGAS_JUDGE_MODEL,
    api_key=settings.litellm_api_key,
    base_url=settings.litellm_base_url.rstrip("/") + "/v1",
    temperature=0,
))
judge_embeddings = LangchainEmbeddingsWrapper(_MiniLMLangchain())

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    AnswerCorrectness(),
    NonLLMContextPrecisionWithReference(),
    NonLLMContextRecall(),
]
run_cfg = RunConfig(max_workers=4, timeout=240)

print(f"judge: {RAGAS_JUDGE_MODEL} | metrics: {[m.name for m in metrics]}")

judge: claude-sonnet-5 | metrics: ['faithfulness', 'answer_relevancy', 'answer_correctness', 'non_llm_context_precision_with_reference', 'non_llm_context_recall']


In [9]:
progress(f"RAGAS scoring OURS: {len(ours_samples)} samples, metrics={[m.name for m in metrics]}...")
ours_dataset = EvaluationDataset.from_list(ours_samples)
ours_result = evaluate(dataset=ours_dataset, metrics=metrics, llm=judge_llm,
                       embeddings=judge_embeddings, run_config=run_cfg)
print("=" * 100)
print("OUR PIPELINE — RAGAS scores")
print("=" * 100)
print(ours_result)

ours_df = ours_result.to_pandas()
ours_df.to_csv("eval/khub_compare_ours_results.csv", index=False)
print("\nsaved -> eval/khub_compare_ours_results.csv")

[09:39:09] RAGAS scoring OURS: 20 samples, metrics=['faithfulness', 'answer_relevancy', 'answer_correctness', 'non_llm_context_precision_with_reference', 'non_llm_context_recall']...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

Exception raised in Job[20]: TimeoutError()


OUR PIPELINE — RAGAS scores
{'faithfulness': 0.8417, 'answer_relevancy': 0.6434, 'answer_correctness': 0.5730, 'non_llm_context_precision_with_reference': 0.2158, 'non_llm_context_recall': 0.2417}

saved -> eval/khub_compare_ours_results.csv


In [10]:
progress(f"RAGAS scoring KHUB: {len(khub_samples)} samples, metrics={[m.name for m in metrics]}...")
khub_dataset = EvaluationDataset.from_list(khub_samples)
khub_result = evaluate(dataset=khub_dataset, metrics=metrics, llm=judge_llm,
                       embeddings=judge_embeddings, run_config=run_cfg)
print("=" * 100)
print("KHUB — RAGAS scores")
print("=" * 100)
print(khub_result)

khub_df = khub_result.to_pandas()
khub_df.to_csv("eval/khub_compare_khub_results.csv", index=False)
print("\nsaved -> eval/khub_compare_khub_results.csv")

[09:46:25] RAGAS scoring KHUB: 20 samples, metrics=['faithfulness', 'answer_relevancy', 'answer_correctness', 'non_llm_context_precision_with_reference', 'non_llm_context_recall']...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

Exception raised in Job[71]: TimeoutError()


Exception raised in Job[72]: TimeoutError()


KHUB — RAGAS scores
{'faithfulness': 0.8073, 'answer_relevancy': 0.5873, 'answer_correctness': 0.6014, 'non_llm_context_precision_with_reference': 0.2583, 'non_llm_context_recall': 0.3500}

saved -> eval/khub_compare_khub_results.csv


## Side-by-side comparison

In [11]:
import pandas as pd

def _metric_means(df):
    return df.select_dtypes(include="number").mean()

ours_means = _metric_means(ours_df)
khub_means = _metric_means(khub_df)

comparison = pd.DataFrame({"ours": ours_means, "khub": khub_means})
comparison["delta (ours - khub)"] = comparison["ours"] - comparison["khub"]
comparison = comparison.round(4)
print(comparison.to_string())

comparison.to_csv("eval/khub_compare_summary.csv")
print("\nsaved -> eval/khub_compare_summary.csv")

                                            ours    khub  delta (ours - khub)
faithfulness                              0.8417  0.8073               0.0344
answer_relevancy                          0.6434  0.5873               0.0561
answer_correctness                        0.5730  0.6014              -0.0284
non_llm_context_precision_with_reference  0.2158  0.2583              -0.0425
non_llm_context_recall                    0.2417  0.3500              -0.1083

saved -> eval/khub_compare_summary.csv


In [12]:
# --- Honest read of the deltas: no overall "winner" declared automatically ---
NOTABLE = 0.05
for metric, row in comparison.iterrows():
    delta = row["delta (ours - khub)"]
    if abs(delta) < NOTABLE:
        verdict = "roughly tied"
    elif delta > 0:
        verdict = f"ours ahead by {delta:.3f}"
    else:
        verdict = f"khub ahead by {abs(delta):.3f}"
    print(f"{metric:45} ours={row['ours']:.3f}  khub={row['khub']:.3f}  -> {verdict}")

print(
    "\nCaveats when reading this table:\n"
    "- khub's retrieved_contexts come from its `snippet` field, which may be a truncated view\n"
    "  of the underlying chunk rather than the full passage our pipeline returns — this can\n"
    "  understate khub's context-precision/recall scores relative to faithfulness/correctness.\n"
    "- khub answers with its own internally-chosen model/strategy per question (no model param\n"
    "  on /api/v1/ask); ours always uses the configured chat_model. Both are evaluated as a real\n"
    "  user would experience them, not on equalized internals.\n"
    "- This is a snapshot: khub is a live, evolving system, and this run reflects its behavior\n"
    "  at the time this notebook executed."
)

faithfulness                                  ours=0.842  khub=0.807  -> roughly tied
answer_relevancy                              ours=0.643  khub=0.587  -> ours ahead by 0.056
answer_correctness                            ours=0.573  khub=0.601  -> roughly tied
non_llm_context_precision_with_reference      ours=0.216  khub=0.258  -> roughly tied
non_llm_context_recall                        ours=0.242  khub=0.350  -> khub ahead by 0.108

Caveats when reading this table:
- khub's retrieved_contexts come from its `snippet` field, which may be a truncated view
  of the underlying chunk rather than the full passage our pipeline returns — this can
  understate khub's context-precision/recall scores relative to faithfulness/correctness.
- khub answers with its own internally-chosen model/strategy per question (no model param
  on /api/v1/ask); ours always uses the configured chat_model. Both are evaluated as a real
  user would experience them, not on equalized internals.
- This is a snap

## Phase 6 — Cleanup: remove our test files from khub

khub is a shared, in-use system — we uploaded 9 files into its live document store for this eval
and now remove them, restoring it to its original 42-document state. (Per khub's functional spec,
only uploaded documents are deletable via the API; the source-of-record library documents are
not, so this cannot touch the real 42.)

In [13]:
for family in uploaded_families:
    r = khub.delete(f"/api/v1/documents/{family}")
    progress(f"  deleted {family} -> HTTP {r.status_code}")

r = khub.get("/api/v1/documents")
r.raise_for_status()
khub_docs_after = r.json()["documents"]
print(f"\nkhub library documents after cleanup: {len(khub_docs_after)} (before eval: {len(khub_docs_before)})")
assert len(khub_docs_after) == len(khub_docs_before), "cleanup did not fully restore khub's original document count — investigate before leaving this run"
khub.close()
progress("khub restored to its pre-eval state.")

[09:55:47]   deleted 1 -> HTTP 200


[09:55:47]   deleted 2 -> HTTP 200


[09:55:47]   deleted 3 -> HTTP 200


[09:55:48]   deleted 4 -> HTTP 200


[09:55:48]   deleted 5 -> HTTP 200


...

[09:55:49]   deleted 7 -> HTTP 200


[09:55:50]   deleted 8 -> HTTP 200


[09:55:50]   deleted test -> HTTP 200



khub library documents after cleanup: 44 (before eval: 42)


AssertionError: cleanup did not fully restore khub's original document count — investigate before leaving this run